# Jakarta Weather Visualization - Calendar Heatmap

This notebook creates a calendar heatmap visualization similar to the reference image, showing temperature data over multiple years.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import calendar
from datetime import datetime

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Read the data
df = pd.read_csv('jakarta_weather_data.csv')
df['time'] = pd.to_datetime(df['time'])
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day

# Display basic info
print(f"Data range: {df['time'].min().date()} to {df['time'].max().date()}")
print(f"\nAvailable temperature columns: tavg, tmin, tmax")
print(f"\nTemperature statistics (Celsius):")
df[['tavg', 'tmin', 'tmax']].describe()

In [ ]:
# Convert temperatures to Fahrenheit
df['tavg_f'] = df['tavg'] * 9/5 + 32
df['tmin_f'] = df['tmin'] * 9/5 + 32
df['tmax_f'] = df['tmax'] * 9/5 + 32

print("Temperature statistics (Fahrenheit):")
df[['tavg_f', 'tmin_f', 'tmax_f']].describe()

## Customize Your Visualization

Change these parameters to customize the heatmap:

In [ ]:
# CUSTOMIZATION PARAMETERS
START_YEAR = 2015
END_YEAR = 2025

# Temperature column options: 'tavg_f', 'tmin_f', 'tmax_f' (Fahrenheit)
#                            or 'tavg', 'tmin', 'tmax' (Celsius)
TEMP_COLUMN = 'tavg_f'

# Display unit
UNIT = '°F'  # or '°C'

# Temperature bins for Fahrenheit
TEMP_BINS_F = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120]
LEGEND_LABELS_F = ['Below 20°F', '20-30°F', '30-40°F', '40-50°F', '50-60°F',
                   '60-70°F', '70-80°F', '80-90°F', '90-100°F', '100+°F']

# Temperature bins for Celsius
TEMP_BINS_C = [-30, -7, -1, 4, 10, 16, 21, 27, 32, 38, 50]
LEGEND_LABELS_C = ['Below -7°C', '-7-(-1)°C', '-1-4°C', '4-10°C', '10-16°C',
                   '16-21°C', '21-27°C', '27-32°C', '32-38°C', '38+°C']

# Color scheme (blue to red)
COLORS = ['#08519c', '#3182bd', '#6baed6', '#9ecae1', '#c6dbef', 
          '#fee0d2', '#fc9272', '#de2d26', '#a50f15']

In [ ]:
# Select appropriate bins and labels
if UNIT == '°F':
    temp_bins = TEMP_BINS_F
    legend_labels = LEGEND_LABELS_F
else:
    temp_bins = TEMP_BINS_C
    legend_labels = LEGEND_LABELS_C

# Filter data by year range
df_filtered = df[(df['year'] >= START_YEAR) & (df['year'] <= END_YEAR)].copy()

## Generate Calendar Heatmap

In [ ]:
# Create the visualization
def create_calendar_heatmap(df, start_year, end_year, temp_col, unit, 
                           temp_bins, colors, legend_labels):
    
    fig = plt.figure(figsize=(28, 16))
    ax = fig.add_subplot(111)
    
    months = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
    
    years = list(range(start_year, end_year + 1))
    cell_height = 0.15
    
    # Year labels
    for year_idx, year in enumerate(years):
        x_center = year_idx * 6.5 + 3
        ax.text(x_center, len(months) + 0.5, str(year), 
                ha='center', va='bottom', fontsize=11, weight='normal', color='#666')
    
    # Draw calendar
    for month_idx, month in enumerate(months):
        y_pos = len(months) - month_idx - 1
        ax.text(-1, y_pos, month, ha='right', va='center', 
                fontsize=11, weight='normal', color='#666')
        
        for year_idx, year in enumerate(years):
            month_data = df[(df['year'] == year) & (df['month'] == month_idx + 1)]
            days_in_month = calendar.monthrange(year, month_idx + 1)[1]
            
            for day in range(1, days_in_month + 1):
                day_data = month_data[month_data['day'] == day]
                
                if len(day_data) > 0 and pd.notna(day_data[temp_col].values[0]):
                    temp = day_data[temp_col].values[0]
                    color_idx = np.digitize(temp, temp_bins) - 1
                    color_idx = max(0, min(color_idx, len(colors) - 1))
                    color = colors[color_idx]
                else:
                    color = '#f0f0f0'
                
                col = (day - 1) % 7
                row = (day - 1) // 7
                
                x = year_idx * 6.5 + col * 0.9
                y = y_pos - row * 0.18
                
                rect = mpatches.Rectangle((x, y), 0.85, cell_height, 
                                         facecolor=color, 
                                         edgecolor='white', 
                                         linewidth=0.5)
                ax.add_patch(rect)
    
    # Styling
    ax.set_xlim(-2, len(years) * 6.5)
    ax.set_ylim(-1, len(months) + 1)
    ax.axis('off')
    
    fig.suptitle("Jakarta's Weather Over the Years", 
                 fontsize=28, weight='bold', y=0.98)
    
    controls_text = f"Start Year: {start_year}    End Year: {end_year}    Temperature Type: Average    Unit: {unit}"
    ax.text(len(years) * 3.25, len(months) + 1.2, controls_text,
            ha='center', va='top', fontsize=11, color='#666')
    
    # Legend
    legend_elements = [mpatches.Patch(facecolor=colors[i], 
                                     edgecolor='none', 
                                     label=legend_labels[i]) 
                      for i in range(len(colors))]
    
    ax.legend(handles=legend_elements, 
              loc='upper center', 
              bbox_to_anchor=(0.5, -0.02),
              ncol=10, 
              frameon=False, 
              fontsize=10)
    
    ax.text(len(years) * 3.25, -1.5, 'Using Jakarta weather data',
            ha='center', va='top', fontsize=10, color='#999')
    
    plt.tight_layout()
    return fig

# Create the plot
fig = create_calendar_heatmap(df_filtered, START_YEAR, END_YEAR, 
                             TEMP_COLUMN, UNIT, temp_bins, 
                             COLORS, legend_labels)

# Save the figure
output_file = f'jakarta_weather_{START_YEAR}_{END_YEAR}.png'
fig.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Saved to: {output_file}")

plt.show()

## Data Analysis

In [ ]:
# Analyze temperature trends
yearly_avg = df.groupby('year')['tavg'].mean()

plt.figure(figsize=(12, 6))
plt.plot(yearly_avg.index, yearly_avg.values, marker='o', linewidth=2, markersize=8)
plt.title('Average Annual Temperature in Jakarta', fontsize=16, weight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average Temperature (°C)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nYearly Average Temperatures:")
print(yearly_avg)

In [ ]:
# Monthly patterns
monthly_avg = df.groupby('month')['tavg'].mean()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(12, 6))
plt.bar(range(1, 13), monthly_avg.values, color='#3498db', alpha=0.8)
plt.title('Average Temperature by Month', fontsize=16, weight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Temperature (°C)', fontsize=12)
plt.xticks(range(1, 13), month_names)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nMonthly Average Temperatures:")
for i, temp in enumerate(monthly_avg.values, 1):
    print(f"{month_names[i-1]}: {temp:.2f}°C")